# CUDA Cluster-Pool Sparse FFN Oracle

This notebook replaces the old per-token sparse Triton benchmark. That path was the wrong hardware shape: every token got a unique sparse row set, so the GPU did `topk + gather + tiny reductions` while dense FFN stayed a few huge GEMMs.

This notebook tests the next shape:

```text
selector features → token clusters → one candidate neuron pool per cluster → cluster GEMMs
```

It does **not** train, does **not** run MacroV2, and does **not** claim final speed. It answers one clean question:

> Can cluster-shared candidate pools keep most of the SVD sparse-neuron FFN quality while restoring GEMM-shaped execution?

In [ ]:
from __future__ import annotations

import dataclasses
import json
import math
import os
import subprocess
import sys
import time
from pathlib import Path
from typing import Any


def _run(cmd: list[str], *, cwd: Path | None = None) -> None:
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=str(cwd) if cwd else None)


def find_or_clone_repo() -> Path:
    """Locate the repo on the remote kernel, or clone it on Colab.

    VSCode can edit this notebook locally while the kernel runs on Colab. In that
    setup, the remote Python process cannot see local repo files unless they have
    been cloned/synced on the remote filesystem.
    """

    env_root = os.environ.get("SPEEDUP_THINGY_REPO")
    candidates: list[Path] = []
    if env_root:
        candidates.append(Path(env_root).expanduser())
    cwd = Path.cwd()
    candidates.extend([
        cwd,
        cwd.parent,
        Path("/content/Speedup-Thingy"),
        Path("/workspace/Speedup-Thingy"),
        Path.home() / "Speedup-Thingy",
    ])
    for candidate in candidates:
        if (candidate / "src" / "recursive_training_engine").exists():
            return candidate.resolve()

    clone_dir = Path(os.environ.get("SPEEDUP_THINGY_CLONE_DIR", "/content/Speedup-Thingy"))
    repo_url = os.environ.get(
        "SPEEDUP_THINGY_REPO_URL",
        "https://github.com/BrownHujay/Speedup-Thingy.git",
    )
    repo_ref = os.environ.get("SPEEDUP_THINGY_REF", "master")
    clone_dir.parent.mkdir(parents=True, exist_ok=True)
    if not clone_dir.exists():
        _run(["git", "clone", "--depth", "1", "--branch", repo_ref, repo_url, str(clone_dir)])
    elif (clone_dir / ".git").exists():
        _run(["git", "fetch", "origin", repo_ref], cwd=clone_dir)
        _run(["git", "checkout", repo_ref], cwd=clone_dir)
        _run(["git", "pull", "--ff-only"], cwd=clone_dir)
    return clone_dir.resolve()


repo_root = find_or_clone_repo()
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

try:
    import recursive_training_engine  # noqa: F401
except ModuleNotFoundError:
    _run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_root), "--no-deps"])

import torch
import torch.nn.functional as F

from recursive_training_engine.config import load_config
from recursive_training_engine.data import load_token_streams
from recursive_training_engine.models import DenseModel
from recursive_training_engine.utils import set_seed

try:
    from recursive_training_engine.cli import cmd_deferred_neuron_cluster_pool_oracle  # noqa: F401
    HAS_CLUSTER_POOL_CLI = True
except Exception:
    HAS_CLUSTER_POOL_CLI = False

print("repo_root:", repo_root)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
print("cluster-pool CLI available:", HAS_CLUSTER_POOL_CLI)

## Settings

Defaults are deliberately sane: one eval batch, no training. Increase `EVAL_BATCHES` after the smoke result makes sense. Set `EVAL_BATCHES = None` only when you really want the config's full eval-token lane.

`DENSE_CHECKPOINT` must point to the dense checkpoint on the remote Colab filesystem. If your checkpoint is elsewhere, edit that one line.

In [ ]:
CONFIG = repo_root / "configs/proof_filter_depth4_dense.yaml"
DENSE_CHECKPOINT = repo_root / "runs/proof_d4_dense/checkpoint.pt"

# Safe default. Set to None for config.data.eval_tokens, or a larger int for a sweep.
EVAL_BATCHES: int | None = 1
BATCH_SIZE: int | None = None

# Cluster-pool grid. This is the new test. M=96 is included because it is a useful smaller pool check.
RANKS = [64]
CLUSTERS = [16, 32]
CANDIDATE_M = [96, 128, 192]
REFERENCE_K = 64
SCORE_MODES = ["sum"]       # choices: "sum", "upgate", "product"
AGGREGATIONS = ["mean"]     # choices: "mean", "max"
CLUSTER_ITERS = 6
TEMPERATURE = 2.0
SEED = 123
COMPUTE_RECALL = True       # Set False for faster quality-only sweeps.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_grad_enabled(False)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True

print(json.dumps({
    "config": str(CONFIG),
    "dense_checkpoint": str(DENSE_CHECKPOINT),
    "eval_batches": EVAL_BATCHES,
    "batch_size": BATCH_SIZE,
    "ranks": RANKS,
    "clusters": CLUSTERS,
    "candidate_m": CANDIDATE_M,
    "reference_k": REFERENCE_K,
    "score_modes": SCORE_MODES,
    "aggregations": AGGREGATIONS,
    "cluster_iters": CLUSTER_ITERS,
    "device": str(DEVICE),
}, indent=2))

## Notebook-Local Cluster Oracle

This cell is self-contained so the notebook still runs even if the remote Colab clone has not picked up the newest CLI command yet. It uses repo model/data classes, then implements the cluster-pool FFN logic here.

In [ ]:
def model_for_dense(config):
    training = dataclasses.replace(config.training, mode="dense_exact")
    model = dataclasses.replace(config.model, topology="dense")
    return dataclasses.replace(config, training=training, model=model)


def load_dense_model(config, checkpoint: Path, device: torch.device) -> DenseModel:
    if not checkpoint.exists():
        raise FileNotFoundError(
            f"Dense checkpoint not found on the remote runtime: {checkpoint}\n"
            "Set DENSE_CHECKPOINT to the Colab path that contains checkpoint.pt."
        )
    model = DenseModel(dataclasses.replace(config.model, topology="dense")).to(device)
    payload = torch.load(checkpoint, map_location=device, weights_only=False)
    model.load_state_dict(payload["model"], strict=True)
    model.eval()
    return model


@torch.no_grad()
def build_svd_factor_cache(dense: DenseModel, *, max_rank: int, device: torch.device):
    factors = []
    for block in dense.blocks:
        weight = block.mlp.wug.weight.detach().float().cpu()
        up_w, gate_w = weight.chunk(2, dim=0)
        layer = {}
        for name, mat in (("up", up_w.t().contiguous()), ("gate", gate_w.t().contiguous())):
            u, s, vh = torch.linalg.svd(mat, full_matrices=False)
            rank = min(max_rank, s.numel())
            layer[f"{name}_a"] = (u[:, :rank] * s[:rank]).to(device=device)
            layer[f"{name}_b"] = vh[:rank, :].to(device=device)
        factors.append(layer)
    return factors


def cluster_assignments_from_features(features: torch.Tensor, *, cluster_count: int, iters: int) -> torch.Tensor:
    token_count = features.shape[0]
    cluster_count = min(max(1, int(cluster_count)), max(1, token_count))
    if cluster_count == 1 or token_count == 1:
        return torch.zeros(token_count, device=features.device, dtype=torch.long)
    x = F.normalize(features.float(), dim=-1)
    init_idx = torch.linspace(0, token_count - 1, steps=cluster_count, device=features.device).round().long()
    centers = x.index_select(0, init_idx).contiguous()
    assignments = torch.zeros(token_count, device=features.device, dtype=torch.long)
    for _ in range(max(1, int(iters))):
        assignments = (x @ centers.t()).argmax(dim=-1)
        new_centers = torch.zeros_like(centers)
        counts = torch.bincount(assignments, minlength=cluster_count).to(new_centers.dtype)
        new_centers.index_add_(0, assignments, x)
        nonempty = counts > 0
        new_centers[nonempty] = new_centers[nonempty] / counts[nonempty].unsqueeze(-1).clamp_min(1.0)
        new_centers[~nonempty] = centers[~nonempty]
        centers = F.normalize(new_centers, dim=-1)
    return assignments


def aggregate_cluster_scores(scores: torch.Tensor, assignments: torch.Tensor, *, cluster_count: int, aggregation: str) -> torch.Tensor:
    if aggregation == "mean":
        agg = scores.new_zeros(cluster_count, scores.shape[-1])
        agg.index_add_(0, assignments, scores)
        counts = torch.bincount(assignments, minlength=cluster_count).to(scores.dtype).clamp_min(1.0)
        return agg / counts.unsqueeze(-1)
    if aggregation == "max":
        rows = []
        for cluster_idx in range(cluster_count):
            mask = assignments == cluster_idx
            rows.append(scores[mask].amax(dim=0) if bool(mask.any()) else scores.new_zeros(scores.shape[-1]))
        return torch.stack(rows, dim=0)
    raise ValueError(f"unsupported aggregation: {aggregation}")


@torch.no_grad()
def cluster_pool_ffn_output(
    block,
    normed: torch.Tensor,
    factors: dict[str, torch.Tensor],
    *,
    rank: int,
    cluster_count: int,
    candidate_m: int,
    reference_k: int,
    score_mode: str,
    aggregation: str,
    cluster_iters: int,
    compute_recall: bool,
):
    original_shape = normed.shape[:-1]
    flat = normed.reshape(-1, normed.shape[-1])
    hidden = block.mlp.wd.in_features
    rank = min(max(1, int(rank)), factors["up_a"].shape[1], factors["gate_a"].shape[1])
    candidate_m = min(max(1, int(candidate_m)), hidden)
    reference_k = min(max(1, int(reference_k)), hidden)
    cluster_count = min(max(1, int(cluster_count)), flat.shape[0])

    q_up = flat @ factors["up_a"][:, :rank]
    q_gate = flat @ factors["gate_a"][:, :rank]
    up_hat = q_up @ factors["up_b"][:rank, :]
    gate_hat = q_gate @ factors["gate_b"][:rank, :]
    gate_hat_act = F.silu(gate_hat)

    wd_rows = block.mlp.wd.weight.t().contiguous()
    wd_norm = block.mlp.wd.weight.detach().norm(dim=0).to(device=flat.device, dtype=flat.dtype)
    up_scores = up_hat.detach().abs() * wd_norm
    gate_scores = gate_hat_act.detach().abs() * wd_norm
    product_scores = (up_hat * gate_hat_act).detach().abs() * wd_norm
    if score_mode == "sum":
        pool_scores = up_scores + gate_scores + product_scores
    elif score_mode == "upgate":
        pool_scores = up_scores + gate_scores
    elif score_mode == "product":
        pool_scores = product_scores
    else:
        raise ValueError(f"unsupported score_mode: {score_mode}")

    features = torch.cat([q_up, q_gate], dim=-1)
    assignments = cluster_assignments_from_features(features, cluster_count=cluster_count, iters=cluster_iters)
    aggregate_scores = aggregate_cluster_scores(
        pool_scores.float(),
        assignments,
        cluster_count=cluster_count,
        aggregation=aggregation,
    )
    candidate_ids = torch.topk(aggregate_scores, k=candidate_m, dim=-1).indices

    out = flat.new_zeros(flat.shape[0], flat.shape[-1])
    up_weight, gate_weight = block.mlp.wug.weight.detach().chunk(2, dim=0)
    cluster_sizes = torch.bincount(assignments, minlength=cluster_count)
    for cluster_idx in range(cluster_count):
        token_idx = torch.nonzero(assignments == cluster_idx, as_tuple=False).flatten()
        if token_idx.numel() == 0:
            continue
        ids = candidate_ids[cluster_idx]
        x_cluster = flat.index_select(0, token_idx)
        up = x_cluster @ up_weight.index_select(0, ids).t().contiguous()
        gate = x_cluster @ gate_weight.index_select(0, ids).t().contiguous()
        z = up * F.silu(gate)
        out.index_copy_(0, token_idx, z @ wd_rows.index_select(0, ids).contiguous())

    aux = {
        "candidate_size_sum": float(candidate_m * flat.shape[0]),
        "selection_count": float(flat.shape[0]),
        "cluster_count_sum": float(cluster_count),
        "nonempty_cluster_count_sum": float((cluster_sizes > 0).sum().detach().cpu()),
        "empty_cluster_count_sum": float((cluster_sizes == 0).sum().detach().cpu()),
        "cluster_pool_ffn_flop_ratio_sum": float(candidate_m / hidden),
        "cluster_metric_count": 1.0,
    }
    nonempty = cluster_sizes[cluster_sizes > 0].float()
    if nonempty.numel():
        aux.update({
            "max_cluster_size_sum": float(nonempty.max().detach().cpu()),
            "min_cluster_size_sum": float(nonempty.min().detach().cpu()),
            "mean_cluster_size_sum": float(nonempty.mean().detach().cpu()),
            "cluster_imbalance_sum": float((nonempty.max() / nonempty.mean().clamp_min(1.0)).detach().cpu()),
        })

    if compute_recall:
        up, gate = block.mlp.wug(normed).chunk(2, dim=-1)
        exact_scores = (up * F.silu(gate)).reshape(flat.shape[0], hidden).detach().abs() * wd_norm
        true_ids = torch.topk(exact_scores, k=reference_k, dim=-1).indices
        token_candidates = candidate_ids.index_select(0, assignments)
        candidate_hits = true_ids.unsqueeze(-1).eq(token_candidates.unsqueeze(1)).any(dim=-1)
        candidate_recall = candidate_hits.float().sum(dim=-1) / max(reference_k, 1)
        candidate_scores = torch.gather(exact_scores, dim=-1, index=token_candidates).sum(dim=-1)
        true_scores = torch.gather(exact_scores, dim=-1, index=true_ids).sum(dim=-1).clamp_min(1e-12)
        aux.update({
            "candidate_recall_sum": float(candidate_recall.sum().detach().cpu()),
            "score_retention_sum": float((candidate_scores / true_scores).sum().detach().cpu()),
        })

    return out.view(*original_shape, -1), aux


@torch.no_grad()
def cluster_pool_full_stack_logits(dense, svd_factors, tokens, *, rank, cluster_count, candidate_m, reference_k, score_mode, aggregation, cluster_iters, compute_recall):
    x = dense.embed(tokens)
    aux: dict[str, float] = {}
    for layer_idx, block in enumerate(dense.blocks):
        u = x + block.attn(block.norm1(x))
        ffn, layer_aux = cluster_pool_ffn_output(
            block,
            block.norm2(u),
            svd_factors[layer_idx],
            rank=rank,
            cluster_count=cluster_count,
            candidate_m=candidate_m,
            reference_k=reference_k,
            score_mode=score_mode,
            aggregation=aggregation,
            cluster_iters=cluster_iters,
            compute_recall=compute_recall,
        )
        for key, value in layer_aux.items():
            aux[key] = aux.get(key, 0.0) + float(value)
        x = u + ffn
    hidden = dense.final_norm(x)
    return hidden @ dense.vocab_weight.t(), hidden, aux


def finish_row(bucket):
    tokens = max(int(bucket.get("tokens", 0)), 1)
    samples = max(int(bucket.get("samples", 0)), 1)
    row = {
        "variant": bucket["variant"],
        "nll_per_token": bucket["loss_sum"] / tokens,
        "kl_to_dense": bucket.get("kl_sum", 0.0) / tokens,
        "final_hidden_cosine": bucket.get("final_hidden_cosine_sum", 0.0) / samples,
        "tokens": tokens,
    }
    for key in ("rank", "clusters", "candidate_m", "score_mode", "aggregation"):
        if key in bucket:
            row[key] = bucket[key]
    count = max(float(bucket.get("selection_count", 0.0)), 1.0)
    metric_count = max(float(bucket.get("cluster_metric_count", 0.0)), 1.0)
    if bucket.get("selection_count", 0.0):
        row["avg_candidate_size"] = bucket["candidate_size_sum"] / count
        if bucket.get("candidate_recall_sum") is not None:
            row["candidate_recall"] = bucket.get("candidate_recall_sum", 0.0) / count
            row["score_retention"] = bucket.get("score_retention_sum", 0.0) / count
    if bucket.get("cluster_metric_count", 0.0):
        row["avg_nonempty_clusters"] = bucket.get("nonempty_cluster_count_sum", 0.0) / metric_count
        row["avg_empty_clusters"] = bucket.get("empty_cluster_count_sum", 0.0) / metric_count
        row["avg_max_cluster_size"] = bucket.get("max_cluster_size_sum", 0.0) / metric_count
        row["avg_min_cluster_size"] = bucket.get("min_cluster_size_sum", 0.0) / metric_count
        row["avg_cluster_imbalance"] = bucket.get("cluster_imbalance_sum", 0.0) / metric_count
        row["estimated_cluster_ffn_flop_ratio"] = bucket.get("cluster_pool_ffn_flop_ratio_sum", 0.0) / metric_count
    if bucket.get("seconds", 0.0):
        row["seconds"] = bucket["seconds"]
        row["tokens_per_second"] = tokens / bucket["seconds"]
    return row

## Load Dense Checkpoint

This is the only place that needs the trained dense checkpoint. The notebook intentionally stops here with a clear error if the checkpoint is not present on the remote runtime.

In [ ]:
if not CONFIG.exists():
    raise FileNotFoundError(f"Config not found: {CONFIG}")
if not DENSE_CHECKPOINT.exists():
    candidates = sorted(repo_root.glob("runs/**/checkpoint.pt"))[:20]
    print("checkpoint candidates under repo_root:", [str(x) for x in candidates])
    raise FileNotFoundError(
        f"Dense checkpoint not found: {DENSE_CHECKPOINT}\n"
        "Edit DENSE_CHECKPOINT above to the checkpoint path on the remote Colab filesystem."
    )

config = model_for_dense(load_config(str(CONFIG)))
if BATCH_SIZE is not None:
    config = dataclasses.replace(
        config,
        training=dataclasses.replace(config.training, batch_size=int(BATCH_SIZE)),
    )
set_seed(SEED)
streams = load_token_streams(config.data, config.training, config.model.vocab_size)
dense = load_dense_model(config, DENSE_CHECKPOINT, DEVICE)
dense.eval()
svd_factors = build_svd_factor_cache(dense, max_rank=max(RANKS), device=DEVICE)

tokens_per_batch = config.training.batch_size * config.training.seq_len
if EVAL_BATCHES is None:
    eval_batches = max(1, math.ceil(config.data.eval_tokens / tokens_per_batch))
else:
    eval_batches = int(EVAL_BATCHES)

print(json.dumps({
    "tokens_per_batch": tokens_per_batch,
    "eval_batches": eval_batches,
    "eval_tokens": eval_batches * tokens_per_batch,
    "model": {
        "d_model": config.model.d_model,
        "d_ff": config.model.d_ff,
        "layers": config.model.n_dense_layers,
        "vocab": config.model.vocab_size,
    },
}, indent=2))

## Run Cluster-Pool Oracle

This runs dense attention plus cluster-shared sparse FFNs. Each cluster uses all `M` candidate neurons as clean GEMMs. There is no per-token final top-k in this path.

In [ ]:
variant_specs = []
for rank in RANKS:
    for clusters in CLUSTERS:
        for candidate_m in CANDIDATE_M:
            for score_mode in SCORE_MODES:
                for aggregation in AGGREGATIONS:
                    variant_specs.append({
                        "variant": f"cluster_svd_rank{rank}_c{clusters}_m{candidate_m}_{score_mode}_{aggregation}",
                        "rank": rank,
                        "clusters": clusters,
                        "candidate_m": candidate_m,
                        "score_mode": score_mode,
                        "aggregation": aggregation,
                    })

buckets: dict[str, dict[str, Any]] = {
    "dense": {
        "variant": "dense",
        "tokens": 0,
        "samples": 0,
        "loss_sum": 0.0,
        "kl_sum": 0.0,
        "final_hidden_cosine_sum": 0.0,
        "seconds": 0.0,
    }
}
for spec in variant_specs:
    buckets[spec["variant"]] = {
        **spec,
        "tokens": 0,
        "samples": 0,
        "loss_sum": 0.0,
        "kl_sum": 0.0,
        "final_hidden_cosine_sum": 0.0,
        "candidate_size_sum": 0.0,
        "candidate_recall_sum": 0.0,
        "score_retention_sum": 0.0,
        "selection_count": 0.0,
        "cluster_metric_count": 0.0,
        "nonempty_cluster_count_sum": 0.0,
        "empty_cluster_count_sum": 0.0,
        "max_cluster_size_sum": 0.0,
        "min_cluster_size_sum": 0.0,
        "cluster_imbalance_sum": 0.0,
        "cluster_pool_ffn_flop_ratio_sum": 0.0,
        "seconds": 0.0,
    }

batches = streams.eval_batches(config.training)
for batch_idx in range(eval_batches):
    tokens, targets = next(batches)
    tokens = tokens.to(DEVICE)
    targets = targets.to(DEVICE)
    target_flat = targets.flatten()
    token_count = int(targets.numel())
    sample_count = int(targets.shape[0])

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    dense_out = dense(tokens, targets, return_loss_per_sample=True)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    dense_seconds = time.perf_counter() - t0

    dense_logits = dense_out.logits.detach()
    dense_hidden = dense_out.meta.hidden.detach()
    dense_logp = F.log_softmax(dense_logits / TEMPERATURE, dim=-1)
    buckets["dense"]["tokens"] += token_count
    buckets["dense"]["samples"] += sample_count
    buckets["dense"]["loss_sum"] += float(dense_out.loss_per_sample.sum().detach().cpu())
    buckets["dense"]["final_hidden_cosine_sum"] += float(sample_count)
    buckets["dense"]["seconds"] += dense_seconds

    for spec in variant_specs:
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        logits, hidden, aux = cluster_pool_full_stack_logits(
            dense,
            svd_factors,
            tokens,
            rank=spec["rank"],
            cluster_count=spec["clusters"],
            candidate_m=spec["candidate_m"],
            reference_k=REFERENCE_K,
            score_mode=spec["score_mode"],
            aggregation=spec["aggregation"],
            cluster_iters=CLUSTER_ITERS,
            compute_recall=COMPUTE_RECALL,
        )
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        seconds = time.perf_counter() - t0

        loss = F.cross_entropy(logits.flatten(0, -2), target_flat, reduction="sum")
        cand_logp = F.log_softmax(logits / TEMPERATURE, dim=-1)
        kl = (dense_logp.exp() * (dense_logp - cand_logp)).sum(dim=-1).sum() * (TEMPERATURE * TEMPERATURE)
        final_cos = F.cosine_similarity(hidden.float().flatten(1), dense_hidden.float().flatten(1), dim=-1).sum()

        bucket = buckets[spec["variant"]]
        bucket["tokens"] += token_count
        bucket["samples"] += sample_count
        bucket["loss_sum"] += float(loss.detach().cpu())
        bucket["kl_sum"] += float(kl.detach().cpu())
        bucket["final_hidden_cosine_sum"] += float(final_cos.detach().cpu())
        bucket["seconds"] += seconds
        for key, value in aux.items():
            bucket[key] = bucket.get(key, 0.0) + float(value)

    rows = [finish_row(buckets[name]) for name in buckets]
    print(json.dumps({"event": "batch_done", "batch": batch_idx + 1, "rows": rows}, indent=2))

rows = [finish_row(buckets[name]) for name in buckets]
report = {
    "mode": "notebook_cluster_pool_oracle",
    "config": str(CONFIG),
    "dense_checkpoint": str(DENSE_CHECKPOINT),
    "eval_batches": eval_batches,
    "eval_tokens": eval_batches * tokens_per_batch,
    "reference_k": REFERENCE_K,
    "cluster_iters": CLUSTER_ITERS,
    "compute_recall": COMPUTE_RECALL,
    "attention": "dense",
    "rows": rows,
}
print(json.dumps(report, indent=2))

## CLI Equivalent

If the remote repo has the latest command, this is the equivalent CLI call. The notebook implementation above does not require the CLI to exist.

In [ ]:
cmd = [
    sys.executable,
    "-m",
    "recursive_training_engine.cli",
    "deferred-neuron-cluster-pool-oracle",
    "--config", str(CONFIG),
    "--dense-checkpoint", str(DENSE_CHECKPOINT),
    "--ranks", *map(str, RANKS),
    "--clusters", *map(str, CLUSTERS),
    "--candidate-m", *map(str, CANDIDATE_M),
    "--reference-k", str(REFERENCE_K),
    "--score-modes", *SCORE_MODES,
    "--aggregations", *AGGREGATIONS,
    "--cluster-iters", str(CLUSTER_ITERS),
]
if EVAL_BATCHES is not None:
    cmd += ["--eval-batches", str(EVAL_BATCHES)]
if BATCH_SIZE is not None:
    cmd += ["--batch-size", str(BATCH_SIZE)]
print(" ".join(cmd))
print("cluster-pool CLI available:", HAS_CLUSTER_POOL_CLI)